Setup docling

In [9]:
import os
import torch

# 1. Ponecháme dôležité premenné prostredia pre MPS fallback
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

# 2. TRIK: Oklameme EasyOCR, aby si myslelo, že MPS je dostupné,
# ale pre Docling (ktorý explicitne dostane "cpu") to bude ignorované.
if torch.backends.mps.is_available():
    # EasyOCR niekedy interne kontroluje iba cuda.is_available()
    # alebo rovno inicializuje 'mps' ak je na Macu,
    # tak sa uistíme, že torch hlási pravdu.
    pass

from tqdm import tqdm
import json
from PIL import Image
import io

from docling.datamodel.base_models import DocumentStream
from docling.document_converter import DocumentConverter, ImageFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, EasyOcrOptions, AcceleratorOptions
)

# ════════════════════════════════════════
#  DOCLING INIT
# ════════════════════════════════════════

opts = PdfPipelineOptions()
opts.do_ocr = True

# Odstránili sme neexistujúci parameter 'device' z EasyOcrOptions
opts.ocr_options = EasyOcrOptions(lang=["sk", "en"])

opts.do_table_structure = False
opts.images_scale = 2.0

# Globálny akcelerátor (Layout modely Doclingu) zostáva bezpečne na CPU
opts.accelerator_options = AcceleratorOptions(
    device="cpu",
    num_threads=4
)

converter = DocumentConverter(
    format_options={InputFormat.IMAGE: ImageFormatOption(pipeline_options=opts)}
)

print("Docling model loaded (Layout: CPU | OCR: Automatická detekcia)")

Docling model loaded (Layout: CPU | OCR: Automatická detekcia)


In [11]:
PNG_PATH = "data/DocLayNet/PNG"

DATA_FOLDERS = [
    "data/DocLayNet/train_data_docling",
    "data/DocLayNet/val_data_docling",
    "data/DocLayNet/test_data_docling",
]


# ════════════════════════════════════════
#  DOCLING INIT (jednorazovo)
# ════════════════════════════════════════

opts = PdfPipelineOptions()
opts.do_ocr = True
opts.ocr_options = EasyOcrOptions(lang=["sk", "en"])
opts.do_table_structure = False
opts.images_scale = 1.0
opts.accelerator_options = AcceleratorOptions(device="cpu", num_threads=4)

converter = DocumentConverter(
    format_options={InputFormat.IMAGE: ImageFormatOption(pipeline_options=opts)}
)

print("Docling model loaded")


# ════════════════════════════════════════
#  DOCLING → NODES
# ════════════════════════════════════════

def run_docling_on_image(pil_image):
    # 1. Prevedieme PIL Image na byte-stream v pamäti (napr. ako PNG)
    image_io = io.BytesIO()
    pil_image.save(image_io, format="PNG")
    image_io.seek(0)

    # 2. Zabalíme to do DocumentStream, ktorý Docling vyžaduje
    doc_stream = DocumentStream(
        name="source_image.png",
        stream=image_io
    )

    # 3. Konvertujeme pomocou doc_stream
    result = converter.convert(doc_stream)

    # --- Zvyšok tvojej funkcie zostáva úplne rovnaký ---
    page = result.pages[0]
    pw = page.size.width if page.size else 1.0
    ph = page.size.height if page.size else 1.0

    nodes = []
    layout = getattr(getattr(page, "predictions", None), "layout", None)
    if not layout:
        return nodes

    # cluster id → text
   # cluster id → text
    text_by_cluster = {}

    # Prechádzame priamo clusters, ktoré obsahujú textové bunky (typicky z OCR)
    layout = getattr(getattr(page, "predictions", None), "layout", None)
    if layout:
        for cl in layout.clusters:
            cid = getattr(cl, "id", None)
            if cid is not None:
                # Mnohé verzie Doclingu majú text priamo v bunkách/slovách clusteru
                # Ak má cluster vlastný text alebo bunky, pospájame ich:
                if hasattr(cl, "text") and cl.text:
                    text_by_cluster[cid] = cl.text.strip()
                elif hasattr(cl, "cells"): # alternatívna štruktúra v Doclingu
                    words = [cell.text for cell in cl.cells if getattr(cell, "text", "")]
                    text_by_cluster[cid] = " ".join(words).strip()

    # direct mapping (NO IoA filtering)
    all_labels = sorted({c.label.value if hasattr(c.label, "value") else str(c.label)
                         for c in layout.clusters})
    label_to_id = {l: i for i, l in enumerate(all_labels)}

    for node_id, cl in enumerate(layout.clusters):
        bbox = cl.bbox
        label = cl.label.value if hasattr(cl.label, "value") else str(cl.label)
        conf = float(getattr(cl, "confidence", 0.0))
        cid = getattr(cl, "id", None)

        x1, y1, x2, y2 = bbox.l, bbox.t, bbox.r, bbox.b

        norm_x1 = x1 / pw
        norm_y1 = y1 / ph
        norm_x2 = x2 / pw
        norm_y2 = y2 / ph

        nodes.append({
            "node_id": node_id,
            "label": label,
            "label_id": label_to_id[label],
            "confidence": round(conf, 6),
            "text": text_by_cluster.get(cid, ""),
            "geometry": {
                "absolute_pixel_coords": [int(x1), int(y1), int(x2), int(y2)],
                "normalized_coords": [norm_x1, norm_y1, norm_x2, norm_y2],
                "normalized_center": [
                    (norm_x1 + norm_x2) / 2,
                    (norm_y1 + norm_y2) / 2
                ],
                "normalized_size": [
                    norm_x2 - norm_x1,
                    norm_y2 - norm_y1
                ]
            }
        })

    return nodes


# ════════════════════════════════════════
#  PIPELINE
# ════════════════════════════════════════

for folder in DATA_FOLDERS:

    graph_files = [f for f in os.listdir(folder) if f.endswith(".json")]
    updated = 0
    skipped = 0

    for filename in tqdm(graph_files, desc=folder.split("/")[-1]):

        graph_path = os.path.join(folder, filename)

        with open(graph_path, encoding="utf-8") as f:
            graph = json.load(f)

        if "docling_nodes" in graph:
            skipped += 1
            continue

        img_path = os.path.join(PNG_PATH, graph["file_name"])

        if not os.path.exists(img_path):
            skipped += 1
            continue

        pil_image = Image.open(img_path).convert("RGB")

        graph["docling_nodes"] = run_docling_on_image(pil_image)

        with open(graph_path, "w", encoding="utf-8") as f:
            json.dump(graph, f, ensure_ascii=False, indent=2)

        updated += 1

    print(f"{folder}: {updated} updated | {skipped} skipped")

print("\nDone!")

Docling model loaded


train_data_docling:   0%|          | 26/19106 [01:05<13:24:47,  2.53s/it]


KeyboardInterrupt: 

In [13]:
import os
import json
from tqdm import tqdm

DATA_FOLDERS = [
    "data/DocLayNet/train_data_docling",
    "data/DocLayNet/val_data_docling",
    "data/DocLayNet/test_data_docling",
]

print("Spúšťam čistenie JSON súborov...")

for folder in DATA_FOLDERS:
    if not os.path.exists(folder):
        print(f"Priečinok nenájdený: {folder}")
        continue

    graph_files = [f for f in os.listdir(folder) if f.endswith(".json")]
    cleaned_count = 0

    for filename in tqdm(graph_files, desc=f"Čistím {folder.split('/')[-1]}"):
        graph_path = os.path.join(folder, filename)

        # 1. Načítanie JSONu
        with open(graph_path, "r", encoding="utf-8") as f:
            try:
                graph = json.load(f)
            except json.JSONDecodeError:
                continue

        # 2. Odstránenie kľúča, ak existuje
        if "docling_nodes" in graph:
            del graph["docling_nodes"]

            # 3. Uloženie upraveného súboru späť
            with open(graph_path, "w", encoding="utf-8") as f:
                json.dump(graph, f, ensure_ascii=False, indent=2)

            cleaned_count += 1

    print(f"{folder}: Vyčistených {cleaned_count} z {len(graph_files)} súborov.")

print("\nHovovo! Môžeš znova spustiť Docling pipeline.")

Spúšťam čistenie JSON súborov...


Čistím train_data_docling: 100%|██████████| 19106/19106 [00:00<00:00, 34014.43it/s]


data/DocLayNet/train_data_docling: Vyčistených 0 z 19106 súborov.


Čistím val_data_docling: 100%|██████████| 9239/9239 [00:00<00:00, 29558.82it/s]


data/DocLayNet/val_data_docling: Vyčistených 0 z 9239 súborov.


Čistím test_data_docling: 100%|██████████| 6895/6895 [00:00<00:00, 30858.97it/s]

data/DocLayNet/test_data_docling: Vyčistených 0 z 6895 súborov.

Hovovo! Môžeš znova spustiť Docling pipeline.
